# Project 1: S&P 500 — From ETF Holdings to SEC Filings

**DSC 785-001: Text Analytics — Summer 2026**
**Due: 6/9/2026**
**Groups of 3–5 students**

**Group Members:** *(fill in your names here)*
- Name 1
- Name 2
- Name 3

## Objective

You will build a data pipeline that starts where every analyst starts — downloading the holdings of a major ETF — and extends into the SEC’s EDGAR system to pull the actual content of corporate filings. By the end, you will have scraped live market data, enriched it with SEC metadata, and read primary-source 10-K filings to form your own analytical view of a sector of the US economy.

This project has three parts:
- **Part A** tests your ability to scrape, clean, and analyze data using Python and Pandas.
- **Part B** tests your ability to work with a Python library and a government API to enrich market data with regulatory metadata, and to extract the key sections of a 10-K filing programmatically.
- **Part C** tests your ability to read primary sources and form an analytical view — the skill that separates an analyst from a data processor.

## Setup

In [ ]:
!pip install edgartools -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
from edgar import Company

---
# Part A: ETF Holdings — Web Scraping and Sector EDA

The iShares Core S&P 500 ETF (ticker: IVV) publishes its full holdings as a downloadable CSV. This is real-world financial data used by portfolio managers, quants, and analysts every day.

### A1. Get the IVV Holdings CSV

Go to the iShares website and find the CSV download URL for the "Detailed Holdings and Analytics" of the IVV ETF:

https://www.ishares.com/us/products/239726/ishares-core-sp-500-etf

**You must document how you found the URL.** In your presentation, show:
1. A screenshot or screen recording of the **live iShares page** in your browser
2. How you identified and copied the exact download URL (e.g., browser DevTools → Network tab, or right-click → Copy Link Address on the download button)
3. Where that URL appears in your code

This is a core web scraping skill — knowing how to find the actual data endpoint behind a website. If you cannot explain how you got the URL, you did not do the scraping.

Fill in the URL below and load it into a DataFrame. You will need to handle:
- The metadata header rows (hint: `skiprows`)
- The thousands separator in numeric columns (hint: `thousands=','`)
- The disclaimer/footer rows at the bottom

In [ ]:
url = ''  # Find and paste the CSV download URL — you must show how you found this

# Your code to load and clean the DataFrame
# df = pd.read_csv(url, skiprows=???)

### A2. Data Cleaning

The raw CSV needs work before it’s usable:
1. Filter to only equity holdings (remove cash, derivatives, futures rows)
2. Convert `Market Value` and `Weight (%)` to numeric types
3. Check for and handle any missing or malformed data
4. Show `df.info()` and `df.describe()` and comment on what you see

In [ ]:
# Your code: filter to equity only

In [ ]:
# Your code: convert Market Value and Weight to numeric

In [ ]:
# df.info()

In [ ]:
# df.describe()

**Your commentary on what you see in the data:**

*(write here)*

### A3. Sector Pie Charts — Counts vs. Market Value

**Before writing any code**, write down your hypothesis: which sectors do you *expect* to dominate by count vs. by market value? Why? (2–3 sentences. This must be written before you see the output — backfilling a "prediction" after seeing the answer is obvious and will be graded accordingly.)

**YOUR HYPOTHESIS** (write this BEFORE running the code below):

*"I expect ... because ..."*

Create **two** pie charts side by side:
1. Number of stocks per GICS sector (using `value_counts()`)
2. Total market value per GICS sector (using `groupby` + `sum`)

**Visualization quality is graded.** Charts with overlapping labels, unreadable text, truncated legends, or any visual artifact that makes the output unclear to a human reader will lose points. If you cannot read it, neither can I. Use `plt.tight_layout()`, adjust figure size, rotate labels, or use a different chart type if pie slices are too small to label cleanly. This applies to every chart in the project.

In [ ]:
# Pie chart 1: count of stocks per sector

In [ ]:
# Pie chart 2: total market value per sector

**What did you expect vs. what did you find?** Which sectors have fewer stocks but disproportionate market value? What does this tell you about market concentration? Name specific companies driving the difference.

*(write here)*

### A4. Top Holdings and Concentration

**Before running the code**, predict: what percentage of total IVV market value do you think the top 10 stocks hold? Write your guess and your reasoning.

**YOUR PREDICTION:** *"I think the top 10 stocks hold roughly __% because ..."*

1. What percentage of total IVV market value is held by the top 10 stocks? Top 20? Top 50?
2. Plot a cumulative weight chart — x-axis is rank (1 to ~500), y-axis is cumulative weight (%)
3. The Herfindahl-Hirschman Index (HHI) measures market concentration: $HHI = \sum_{i=1}^{N} s_i^2$ where $s_i$ is each stock’s weight as a proportion. Calculate the HHI for IVV by sector. Which sector is most concentrated (dominated by a few names)?

In [ ]:
# Top 10, 20, 50 concentration

In [ ]:
# Cumulative weight chart

In [ ]:
# HHI by sector

**How did your prediction compare? What surprised you?**

*(write here)*

### A5. Exchange Analysis

The CSV includes an `Exchange` column. Analyze:
1. How many stocks trade on NYSE vs NASDAQ vs other exchanges?
2. Is there a pattern — do certain sectors favor one exchange? Why might that be?
3. Create a cross-tabulation (pivot table) of Sector × Exchange

In [ ]:
# Your code: exchange counts

In [ ]:
# Your code: sector x exchange cross-tabulation

**Your analysis:**

*(write here)*

---
# Part B: EDGAR — From Market Data to Regulatory Filings

The IVV CSV tells you what the market thinks (price, weight, sector). SEC EDGAR tells you what companies actually report to regulators — their industry classification, their risk factors, their management’s view of the business. In this part you will use the `edgartools` Python library to pull both metadata and actual filing content from EDGAR.

### About `edgartools`

`edgartools` is a Python library that wraps the SEC EDGAR API. You create a `Company` object from a ticker, and it gives you access to filings, financial facts, and company metadata:

```python
from edgar import Company

company = Company("AAPL")
print(company.name)              # 'APPLE INC'
print(company.sic)               # '3571'
print(company.industry)          # 'Electronic Computers'
print(company.filer_type)        # 'Domestic'
print(company.business_category) # 'Operating Company'
print(company.is_foreign)        # False
print(company.fiscal_year_end)   # '0930' (September 30)
```

**Important:** EDGAR rate-limits requests to 10 per second. Your code must handle this — either use `time.sleep()` or handle exceptions gracefully. Start with a small sample (10–20 tickers) to develop your code, then scale to the full universe.

### B1. Build an EDGAR Profile DataFrame

For each equity ticker in your IVV holdings, use `edgartools` to pull:
- `sic` — the SEC’s Standard Industrial Classification code (4 digits)
- `industry` — the SIC description (e.g., "Electronic Computers")
- `filer_type` — 'Domestic', 'Foreign', or 'Canadian'
- `business_category` — 'Operating Company', 'REIT', 'Bank', 'ETF', etc.
- `is_foreign` — boolean
- `fiscal_year_end` — fiscal year end date

Handle errors gracefully — some tickers may fail (e.g., share classes like BRK.B, or name mismatches). Log failures instead of crashing.

In [ ]:
# Start small for development
sample_tickers = df['Ticker'].head(20).tolist()

edgar_data = []
failures = []

for ticker in sample_tickers:
    try:
        company = Company(ticker)
        edgar_data.append({
            'Ticker': ticker,
            'SIC': company.sic,
            'SIC_Industry': company.industry,
            'Filer_Type': company.filer_type,
            'Business_Category': company.business_category,
            'Is_Foreign': company.is_foreign,
            'Fiscal_Year_End': company.fiscal_year_end
        })
        time.sleep(0.15)  # respect rate limits
    except Exception as e:
        failures.append({'Ticker': ticker, 'Error': str(e)})

edgar_df = pd.DataFrame(edgar_data)
print(f"Success: {len(edgar_data)}, Failures: {len(failures)}")

**Once your code works on the sample, scale to the full ~500 tickers.** This will take several minutes. Save the result so you don’t have to re-run it.

In [ ]:
# Your code to run on full universe
# Save: edgar_df.to_csv('edgar_profiles.csv', index=False)

In [ ]:
# Show your failures log
# pd.DataFrame(failures)

### B2. GICS vs. SIC — Two Ways to Classify an Industry

Now you have two industry classifications for each company: the GICS sector (from the iShares CSV — 11 sectors) and the SIC code (from EDGAR — a 4-digit code with ~1,000 industries). These are different systems built by different organizations for different purposes.

Join your IVV DataFrame with your EDGAR DataFrame on Ticker, then explore the relationship:

1. **SIC is hierarchical.** The first 2 digits of the SIC code define a "Major Group" (~83 groups). Create a new column `SIC_2digit` by extracting the first 2 digits. How many distinct SIC Major Groups exist within each GICS sector? Which GICS sectors are the most internally diverse (i.e., contain the widest range of SIC industries)?

2. **Find the mismatches.** Identify specific companies where the GICS sector and the SIC industry tell noticeably different stories. Some examples to check: Is Visa (V) classified the same way in both systems? What about Tesla (TSLA)? Meta (META)? What about companies that moved during the 2018 GICS reshuffle (Google, Facebook moved from IT to Communication Services — but did their SIC codes change)?

3. **In 2–3 paragraphs, explain:** SIC was created by the US government in 1937 (last major revision 1987). GICS was created by S&P and MSCI in 1999 and is updated regularly. What does each system optimize for? Why does the SEC still use a 1987-era classification? When would you prefer one over the other?

In [ ]:
# Join the DataFrames
# merged_df = pd.merge(df, edgar_df, on='Ticker', how='left')

# Your analysis

In [ ]:
# SIC 2-digit diversity by GICS sector

In [ ]:
# Mismatch examples: V, TSLA, META, GOOGL

**Your 2–3 paragraph explanation of SIC vs. GICS:**

*(write here)*

### B3. Reading a 10-K — The Key Sections

A 10-K annual filing is the most important document a public company produces. It is the primary source for any serious analysis. The `edgartools` library lets you extract individual sections programmatically.

Pick **one company** you find interesting and pull its latest 10-K:

In [ ]:
company = Company("NVDA")  # or your choice
tenk = company.latest_tenk

Now extract and read the following sections **in separate cells** — one cell per section. For each section, print or display the content, then write 2–3 sentences summarizing what you learned.

#### B3a. Item 1 — Business Description

This is the company’s own description of what it does, how it makes money, and how it is organized.

In [ ]:
# Extract Item 1
item1 = tenk['Item 1']
print(item1[:3000])  # Print first 3000 characters — read the rest in the full output

# Your summary: What does this company actually do? How is it organized?

**Your summary of Item 1:**

*(write here)*

#### B3b. Item 1A — Risk Factors

This is the company’s legally required disclosure of everything that could go wrong. It is the most underrated section of a 10-K — it tells you what management is worried about.

In [ ]:
# Extract Item 1A
item1a = tenk['Item 1A']
print(item1a[:3000])

# Your summary: What are the top 3 risks this company discloses?
# Are any of them surprising? Do any feel like boilerplate vs. genuine concern?

**Your summary of Item 1A:**

*(write here)*

#### B3c. Item 7 — Management’s Discussion and Analysis (MD&A)

This is management’s narrative about how the business performed, why results changed, and where things are headed. It is the section analysts read most carefully.

In [ ]:
# Extract Item 7
item7 = tenk['Item 7']
print(item7[:3000])

# Your summary: What is management's tone — optimistic, cautious, defensive?
# What do they highlight as drivers of performance? What do they downplay?

**Your summary of Item 7:**

*(write here)*

#### B3d. Item 8 — Financial Statements (Optional / Extra Credit)

`edgartools` can extract structured financial data. If you want to go further:

In [ ]:
# Get financials
financials = company.get_financials()
if financials:
    income_stmt = financials.income_statement
    print(income_stmt)

# Or get it as a DataFrame:
# company.income_statement(as_dataframe=True)

### B4. Scaling Section Extraction

Now generalize what you did in B3. Write a function that takes a ticker and returns a dictionary of its key 10-K sections (Item 1, Item 1A, Item 7). Run it on 10–20 companies and store the results.

You will need this for Part C — so pick companies from the sector you plan to analyze.

In [ ]:
def extract_10k_sections(ticker):
    """Extract key sections from a company's latest 10-K.
    Returns a dict with keys: ticker, name, item1, item1a, item7
    Returns None if the 10-K cannot be retrieved."""
    try:
        company = Company(ticker)
        tenk = company.latest_tenk
        if tenk is None:
            return None
        return {
            'ticker': ticker,
            'name': company.name,
            'item1': tenk['Item 1'],
            'item1a': tenk['Item 1A'],
            'item7': tenk['Item 7']
        }
    except Exception as e:
        print(f"Failed for {ticker}: {e}")
        return None

# Test on your sector picks
# results = []
# for ticker in my_sector_tickers:
#     result = extract_10k_sections(ticker)
#     if result:
#         results.append(result)
#     time.sleep(0.2)

---
# Part C: Sector Deep Dive — Reading Filings Like an Analyst

This is the analytical heart of the project. You will move from data processing to actual analysis of primary sources.

### C1. Choose Your Sector

Using your merged DataFrame from Parts A and B, choose a sector that interests your group. You can use either:
- A **GICS sector** (e.g., Information Technology, Health Care, Energy), or
- A **SIC-based grouping** if you found a more meaningful cluster in B2 (e.g., SIC 357x "Computer Hardware" or SIC 283x "Drugs")

From that sector, select the **3–5 largest companies by market value**. These are the companies you will analyze in depth.

In [ ]:
# Your code to select your sector and top companies
# Example:
# my_sector = merged_df[merged_df['Sector'] == 'Information Technology']
# top_companies = my_sector.nlargest(5, 'Market Value')

### C2. Your Group’s Own Questions (Required — This Cannot Be Delegated to AI)

Before you begin the comparative analysis below, your group must formulate **3 original questions** that you want to answer by reading the filings. These must be specific, non-obvious questions that arise from your group’s own curiosity about the sector — not generic questions that any LLM could produce.

Examples of **good** questions (specific, driven by genuine curiosity):
- "Broadcom acquired VMware in 2023 — does the 10-K show integration risk that isn’t priced into the stock?"
- "Do any of the semiconductor companies disclose customer concentration risk related to a single hyperscaler?"
- "Energy companies talk about the energy transition — are they actually spending capex on renewables, or is it just language?"

Examples of **bad** questions (generic, easily generated by AI):
- "What are the key risks facing the technology sector?"
- "How do these companies compare in terms of revenue growth?"
- "What is each company’s competitive advantage?"

Write your 3 questions here **before** you begin reading the filings. You will answer them in C4.

**YOUR GROUP’S QUESTIONS:**

1. *(write here)*

2. *(write here)*

3. *(write here)*

### C3. Comparative Filing Analysis

Using the 10-K sections you extracted in B3/B4, read and compare across your 3–5 companies. Present your findings on the following — you do not need an LLM for this. Read the filings yourself.

#### Industry Landscape
Based on the **Item 1 (Business Description)** of each company:
- What do these companies have in common? Where do their business models diverge?
- Are they competing directly, or do they occupy different niches within the sector?
- What is each company’s primary revenue source?

**Your Industry Landscape analysis:**

*(write here)*

#### Risks and Vulnerabilities
Based on the **Item 1A (Risk Factors)** of each company:
- What risks are shared across all companies in this sector? (These are industry-level risks.)
- What risks are unique to specific companies? (These are company-specific risks.)
- Are there any risks that one company discloses prominently but others barely mention? What might explain that?

**Your Risks and Vulnerabilities analysis:**

*(write here)*

#### Management Outlook and Strategy
Based on the **Item 7 (MD&A)** of each company:
- How does each company describe its recent performance — growth, decline, transition?
- What is each company investing in? (Look for mentions of capex, R&D, acquisitions, new markets.)
- Where do the strategies diverge? Is one company betting on a different future than the others?

**Your Management Outlook analysis:**

*(write here)*

### C4. Answer Your Own Questions

Now answer the 3 questions your group wrote in C2. For each question:
1. State the question
2. Cite the specific filing sections, paragraphs, or data points you used to answer it
3. State your answer and whether it surprised you

This section is a significant part of the Part C grade. Generic questions will produce generic answers and receive a generic grade.

**Question 1:**

*(state the question, cite the evidence, give your answer)*

**Question 2:**

*(state the question, cite the evidence, give your answer)*

**Question 3:**

*(state the question, cite the evidence, give your answer)*

### C5. Your Thesis

Based on everything you have read and analyzed, present a thesis about your chosen sector. This should be a clear, defensible view supported by evidence from the filings. Some angles:

- **Industry trends:** What themes appear across all the filings? Is the sector consolidating, fragmenting, pivoting to a new technology, or facing regulatory pressure?
- **Competitive positioning:** Which company appears best positioned, and why? Which appears most vulnerable?
- **Distinguishing strategies:** Are these companies converging on the same strategy, or are they making fundamentally different bets about the future? What does each management team seem to believe that the others do not?

**Every claim must be supported by specific evidence from the filings.** "NVIDIA is investing heavily in AI" is not analysis — anyone can say that. "NVIDIA’s MD&A discloses $X billion in data center capex, a Y% increase YoY, while AMD’s comparable figure is $Z billion" is analysis. Generic claims without specific numbers, quotes, or filing references will receive a C-level grade at best. Specific cross-referencing between filing sections and portfolio data is what earns an A.

**Your Thesis:**

*(write here)*

---
## AI Disclosure (Required)

Every submission must include an `AI_Disclosure` section at the end of your notebook documenting:
1. Which AI tools you used (ChatGPT, Claude, Copilot, etc.)
2. What you used them for (code generation, debugging, analysis, writing)
3. What the AI got wrong and what you had to fix
4. What you verified independently

Remember: you are the senior analyst. The AI is your junior analyst. If you sign your name to it, you own every line — including the ones the AI got wrong.

**AI Tools Used:**

*(fill in)*

**What I used AI for:**

*(fill in)*

**What AI got wrong / what I fixed:**

*(fill in)*

**What I verified independently:**

*(fill in)*

---
## What to Submit

1. **Presentation Video** (5–10 min) — each group member must speak for approximately equal time. Your presentation must include a **live code walkthrough** where you run the notebook cell by cell (or key sections) and show the output generating in real time. A slide deck over static screenshots is not sufficient — I need to see the code execute. If you cannot run the code live, that tells me something.
2. **Google Colab notebook** — with the names of each group member. Set sharing to "Anyone with the link can view." Code must run top-to-bottom without errors when I click "Run All." Any code after the first error will not be graded.
3. **Trello board** — tasks broken into tickets, assigned to members, moved to Done on completion
4. **AI Disclosure** — in the notebook, as described above

---
## Grading Rubric

### Prerequisites (Pass/Fail)
These must be met to receive any credit:
- **Mandatory Presentation Participation**: Every group member must appear and speak. Absent = 0%.
- **Code Submission**: Must include the Colab notebook. No code = 0%.
- **Original Work**: Code must directly address the project tasks. Unmodified AI-generated code that doesn’t integrate with the project requirements = 0%.
- **Live Code Execution**: Presentation must include running the notebook live (cell by cell or key sections). Static screenshots only = significant penalty.

### Scoring (once prerequisites are met)
- **Presentation Quality (30%)**: Clear storytelling, data-driven claims backed by filing evidence, professional delivery, equal participation, live code demo
- **Technical Implementation (30%)**: Code runs without errors, data properly cleaned, EDGAR API calls handled with error logging, section extraction works, visualizations are clear and labeled
- **Analytical Depth (20%)**: Quality of hypotheses vs. findings, quality of group’s original questions (C2) and answers (C4), specificity of filing evidence in thesis (C5). Generic analysis = C. Specific cross-referencing with numbers and filing citations = A.
- **Project Management (10%)**: Trello board shows clear task breakdown, responsibilities assigned, tickets completed
- **Visualization Quality (10%)**: All charts, plots, and tables must be human-readable. Overlapping text, truncated labels, unreadable legends, illegible font sizes, or any visual artifact that obscures information will lose points. If you cannot read it at a glance, fix it before submitting.

### What Separates an A from a B from a C
- **A**: Hypotheses are specific and show genuine prior thinking. Original questions (C2) are sharp and sector-specific. Filing analysis cites specific dollar amounts, percentages, and quotes. Thesis connects portfolio data to filing evidence. Charts are clean and well-labeled. Code runs live without fumbling.
- **B**: Hypotheses are present but vague. Questions are reasonable but could apply to any sector. Filing analysis is accurate but general. Thesis is defensible but lacks specific numbers. Charts are readable. Code runs.
- **C**: Hypotheses are backfilled or missing. Questions are generic ("What are the risks?"). Filing analysis reads like an AI summary without specific evidence. Charts have visual issues. Code has errors or cannot be demonstrated live.

---
## Tips

- **Start with Part A.** It uses skills you should already have (Pandas, scraping, plotting). Get it working first.
- **Write your hypotheses BEFORE running the code.** I can tell when a "prediction" was written after seeing the answer. The point is not to be right — it’s to think before you compute.
- **Part B will be slow.** EDGAR API calls take time. Build your code on 10–20 tickers, then scale. Save intermediate results to CSV so you don’t re-pull every time.
- **Error handling matters.** Not every ticker will work cleanly with `edgartools`. The professional move is to log the failure and keep going, not crash. Your failure log is part of the analysis.
- **Read the filings.** Part C cannot be done well by skimming. The groups that read the actual Item 1, Item 1A, and Item 7 text will produce better analysis than the groups that try to summarize from headlines. This is the skill that matters most.
- **Your original questions (C2) matter.** Generic questions get generic grades. If your questions could apply to any sector without modification, they are not specific enough.
- **Fix your charts.** If text overlaps, labels are cut off, or a legend covers the data — fix it before submitting. Use `figsize`, `tight_layout()`, `autopct` formatting, label rotation, or switch chart types. Unreadable output is unprofessional output.
- **Practice the live demo.** You will run the code in your presentation. If it breaks live, that’s OK — debugging live shows competence. But if you’ve never actually run the code yourself and it fails on every cell, that tells a different story.
- **Part C is what separates an A from a B.** Anyone can make a pie chart. The question is: can you read primary sources, ask your own questions, and form a view that is interesting, true, and supported by specific evidence?
- **Read the `edgartools` documentation:** https://github.com/dgunning/edgartools — there is more available than what this template covers. Explore.